# MultiScale CNEEP 2D Notebook

2D implementation of the MultiScale CNEEP model for spatial entropy production estimation across different correlation lengths (k).

In [ ]:
### for local server ###
import sys
import os

CNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')
if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

In [ ]:
sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from tqdm import tqdm
import matplotlib.pyplot as plt
from generate_trajectories import ActiveModelB

## 1. Hyperparameters

In [ ]:
#
# Hyper parameters (Matched to AMB_1D training parameters)
#
opt = Namespace()
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# alpha-NEEP params
opt.alpha     = -0.5
opt.beta      = 0.0
opt.lam       = 0.0
opt.threshold = 0.01

opt.periodic    = True
opt.positional  = False
opt.latent_size = 10

# training (from 1D)
opt.n_iter           = 10000
opt.train_batch_size = 2048
opt.test_batch_size  = 2048
opt.video_batch_size = 256
opt.lr               = 1e-2
opt.wd               = 1e-8
opt.input_scalar     = 1
opt.loss_scalar      = 1
opt.scalar           = 1
opt.clip_norm        = 1

# MultiScale CNEEP specifically
opt.max_distance = 5
opt.include_k0   = False

opt.record_freq = 100
opt.seed        = 3

# dataset / model architecture
opt.n_layer     = 2
opt.n_channel   = 8
opt.n_hidden    = 2
opt.input_shape = (64, 64)    # AMB grid size 2D
opt.M           = 1000
opt.M_test      = 1
opt.L           = 1000
opt.L_test      = 10000
opt.seq_len     = 2
opt.val_ratio   = 0.2
opt.time_step   = 0.001       # dt

# AMB model parameters
kwargs = dict(
    Lx=64, Ly=64, dx=1.0,
    a=0.25, b=0.25, kappa=4.0,
    lam=20.0, D=1, dt=0.001,
    smooth=True,
    backend='torch',
    use_gpu=True,
    epr_mu_active_only=True,
)
n_steps   = opt.L
burn_in   = 10000
init_mode = 'circle'

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)

# results folder
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(
    result_folder, f"Corr2D-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")


## 2. Generate AMB Trajectories (Train & Test)

In [ ]:
#
# Generate TRAIN trajectories (batch/ensemble)
#
train_seed = 42
np.random.seed(train_seed)
torch.manual_seed(train_seed)

model_amb_train = ActiveModelB(**kwargs)

print(f"[INFO] Generating TRAIN trajectories (M={opt.M}, L={opt.L}, burn_in={burn_in})")
trajectories_train = model_amb_train.generate_trajectories(
    n_trajectories=opt.M,
    n_steps=opt.L,
    burn_in=burn_in,
    init_mode=init_mode,
)
print(f"[INFO] Train trajectories shape: {trajectories_train.shape}")

## 3. Generate TEST Trajectories

In [ ]:
#
# Generate TEST trajectories
#
test_seed = 123
np.random.seed(test_seed)
torch.manual_seed(test_seed)

model_amb_test = ActiveModelB(**kwargs)

print(f"[INFO] Generating TEST trajectories (M={opt.M_test}, L={opt.L_test}, burn_in={burn_in})")
trajectories_test = model_amb_test.generate_trajectories(
    n_trajectories=opt.M_test,
    n_steps=opt.L_test,
    burn_in=burn_in,
    init_mode=init_mode,
)
print(f"[INFO] Test trajectories shape: {trajectories_test.shape}")
traj_test = trajectories_test

## 3. Prepare Video Tensors

In [ ]:
#
# Prepare video tensors: (M, L, 1, Lx, Ly)
#
train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

traj_train_new = trajectories_train[:train_val_split_idx]
traj_val = trajectories_train[train_val_split_idx:]

train_video = torch.from_numpy(traj_train_new).float().to(opt.device)
train_video = train_video.unsqueeze(2)   # (M_train, L, 1, Lx, Ly)

val_video = torch.from_numpy(traj_val).float().to(opt.device)
val_video = val_video.unsqueeze(2)       # (M_val, L, 1, Lx, Ly)

test_video = torch.from_numpy(trajectories_test).float().to(opt.device)
test_video = test_video.unsqueeze(2)     # (M_test, L_test, 1, Lx, Ly)

print(f"Train video tensor: {train_video.shape}")
print(f"Val video tensor:   {val_video.shape}")
print(f"Test video tensor:  {test_video.shape}")

## 4. Train MultiScale CNEEP 2D Model

In [ ]:
from models.NEEP_Corr_2D import MultiScaleCNEEP2D
from livelossplot import PlotLosses

mean = torch.mean(train_video)
std  = torch.std(train_video)
transform = lambda x: (x - mean) * opt.input_scalar / std

model = MultiScaleCNEEP2D(opt).to(opt.device)
optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#
# Training loop (MultiScaleCNEEP2D)
#
# model output: [B, K+1] — EP contribution at each distance k
# total EP per sample = sum over all distances
#
train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

liveloss = PlotLosses()
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None

train_losses = []
valid_losses = []
    best_val_loss = float('inf')
    best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')

for it in tqdm(range(1, opt.n_iter + 1)):
    # ── Train step ──
    model.train()
    batch = next(train_sampler)

    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.cat(slices, dim=1).float().to(opt.device))

    # model forward: [B, K+1]
    J_all = model(x) / opt.scalar   # [B, K+1] EP at each distance

    # Total EP per sample: sum over all correlation distances
    ent_production = J_all.sum(dim=1)  # [B]

    optim.zero_grad()

    # alpha-NEEP loss on total EP
    if opt.alpha == 0:
        loss = (- ent_production + (torch.exp(-ent_production) - 1)).mean()
    else:
        loss = (- (torch.exp(opt.alpha * ent_production) - 1) / opt.alpha
            + (torch.exp(-(1 + opt.alpha) * ent_production) - 1) / (1 + opt.alpha)).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    train_losses.append(loss.item())

    # ── Validation & checkpoint ──
    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))

                vJ = model(vx) / opt.scalar
                v_ep = vJ.sum(dim=1)

                if opt.alpha == 0:
                    vloss = (- v_ep + (torch.exp(-v_ep) - 1)).sum().item()
                else:
                    vloss = (- (torch.exp(opt.alpha * v_ep) - 1) / opt.alpha
                        + (torch.exp(-(1 + opt.alpha) * v_ep) - 1) / (1 + opt.alpha)).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / n_val
        valid_losses.append(avg_val)

        # Save checkpoint
        state = {
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
            'iteration': it,
        }
        torch.save(state, current_checkpoint_path)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        # Update livelossplot
        if smooth_train_loss is None:
            smooth_train_loss = loss.item()
            smooth_val_loss = avg_val
        else:
            smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
            smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val

        liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_val_loss})
        liveloss.send()

print('Training finished.')
print(f'Checkpoint: {current_checkpoint_path}')


## 5. Training Curves

In [ ]:
#
# Training curves
#
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(train_losses[100:]); axes[0].set_title('Train Loss')
axes[1].plot(valid_losses); axes[1].set_title('Valid Loss')
for ax in axes: ax.set_xlabel('Iteration')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/training_curves.png', dpi=150)
plt.show()


## 4-b. Model Selection

In [ ]:
# Load the Best or Final model
load_best = True  # Set to False to load the model from the final iteration

best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')
if load_best and os.path.exists(best_checkpoint_path):
    print("Loading BEST model (lowest validation loss)...")
    checkpoint = torch.load(best_checkpoint_path, map_location=opt.device)
else:
    print("Loading FINAL iteration model...")
    checkpoint = torch.load(current_checkpoint_path, map_location=opt.device)
    
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print(f"Loaded model from iteration {checkpoint.get('iteration', 'Unknown')}")

## 5. Ground Truth EPR vs Predicted EPR

In [ ]:
#
# Ground truth EPR on TEST data
# seq_len에 맞춰 (seq_len-1)개 transition을 하나의 window로 합산
# Corr_1D와 동일한 per-window compute_local_epr_density 방식 사용
#
stride = opt.seq_len - 1  # seq_len=2이면 stride=1 (기존과 동일)
n_windows = (opt.L_test - 1) // stride

print(f"[INFO] Computing GT EPR on TEST data (seq_len={opt.seq_len}, stride={stride}, n_windows={n_windows}) ...")

gt_total_epr = np.zeros(n_windows)
gt_epr_map_sum = np.zeros((kwargs['Lx'], kwargs['Ly']))

traj_test_gpu = torch.tensor(traj_test, dtype=torch.float64).to(opt.device)
for w in tqdm(range(n_windows)):
    t_start = w * stride
    window_epr_map = np.zeros((kwargs['Lx'], kwargs['Ly']))
    for s in range(stride):
        t = t_start + s
        epr_map = model_amb_test.compute_local_epr_density(
            traj_test_gpu[:, t], traj_test_gpu[:, t+1])
        mean_epr_map = epr_map.mean(dim=0)  # average over M ensembles
        window_epr_map += mean_epr_map.cpu().numpy()
    gt_epr_map_sum += window_epr_map
    gt_total_epr[w] = np.sum(window_epr_map) * model_amb_test.dx**2

# Time-averaged GT EPR map for spatial visualization
gt_epr_maps = (gt_epr_map_sum / n_windows)[np.newaxis, ...]  # (1, Lx, Ly)

print(f"GT mean EPR rate (per dt): {(gt_total_epr / stride).mean():.6e}")
print(f"GT windows: {n_windows}")


In [ ]:
#
# Compare total Predicted EP (sum of all J_k) with GT EPR
#
model.eval()
pred_total_ep = []

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar  # [B, K+1]
        total_ep = J_all.sum(dim=1)     # [B]
        pred_total_ep.append(total_ep.cpu().numpy())

pred_total_ep = np.concatenate(pred_total_ep) # [N_total]

min_len = min(len(gt_total_epr), len(pred_total_ep))
dt = kwargs['dt']
time_axis = np.arange(min_len) * dt

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# (a) Instantaneous EPR
axes[0].plot(time_axis, gt_total_epr[:min_len], lw=0.5, alpha=0.6, label='GT EPR')
axes[0].plot(time_axis, pred_total_ep[:min_len] / dt, lw=0.5, alpha=0.6, label='Pred EP/dt')
axes[0].set_ylabel('EPR')
axes[0].legend()
axes[0].set_title('Instantaneous EPR Comparison')

# (b) Cumulative EP
axes[1].plot(time_axis, np.cumsum(gt_total_epr[:min_len] * dt), label='GT Cumul EP')
axes[1].plot(time_axis, np.cumsum(pred_total_ep[:min_len]), label='Pred Cumul EP')
axes[1].set_ylabel('Cumulative EP')
axes[1].legend()

# (c) Running Average
window = 100
gt_smooth = np.convolve(gt_total_epr[:min_len], np.ones(window)/window, mode='same')
pred_smooth = np.convolve(pred_total_ep[:min_len]/dt, np.ones(window)/window, mode='same')
axes[2].plot(time_axis, gt_smooth, label='GT (Smooth)')
axes[2].plot(time_axis, pred_smooth, label='Pred (Smooth)')
axes[2].set_ylabel('EPR (Running Avg)')
axes[2].set_xlabel('Time')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_timeseries.png', dpi=150)
plt.show()

print(f'GT mean EPR:   {gt_total_epr[:min_len].mean():.6e}')
print(f'Pred mean EPR:  {(pred_total_ep[:min_len]/dt).mean():.6e}')

# R^2 Scatter Plot (Window=1 and Window=100)
# Use Pearson correlation squared for best-fit R2
def compute_r2_best_fit(y_true, y_pred):
    corr_matrix = np.corrcoef(y_true, y_pred)
    corr = corr_matrix[0, 1]
    return corr ** 2

gt_raw = gt_total_epr[:min_len]
pred_raw = pred_total_ep[:min_len] / dt

r2_raw = compute_r2_best_fit(gt_raw, pred_raw)

# Use 'valid' convolution for R2 to avoid boundary drop-offs
valid_gt_smooth = np.convolve(gt_raw, np.ones(window)/window, mode='valid')
valid_pred_smooth = np.convolve(pred_raw, np.ones(window)/window, mode='valid')
r2_smooth = compute_r2_best_fit(valid_gt_smooth, valid_pred_smooth)

fig_r2, axes_r2 = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Window=1
axes_r2[0].scatter(gt_raw, pred_raw, alpha=0.3, color='steelblue', s=10)
# y = x line
min_val = min(gt_raw.min(), pred_raw.min())
max_val = max(gt_raw.max(), pred_raw.max())
axes_r2[0].plot([min_val, max_val], [min_val, max_val], 'k--', lw=1, label='y = x')
# Best fit line
m, c = np.polyfit(gt_raw, pred_raw, 1)
axes_r2[0].plot(gt_raw, m * gt_raw + c, 'r-', lw=2, label=f'Fit: y = {m:.2f}x + {c:.2f}')
axes_r2[0].set_title(f'Instantaneous (Window=1)\n$R^2$ (Fit) = {r2_raw:.4f}')
axes_r2[0].set_xlabel('Ground Truth EPR')
axes_r2[0].set_ylabel('Predicted EPR')
axes_r2[0].legend()

# Plot 2: Window=100
axes_r2[1].scatter(valid_gt_smooth, valid_pred_smooth, alpha=0.5, color='darkorange', s=10)
# y = x line
min_val_s = min(valid_gt_smooth.min(), valid_pred_smooth.min())
max_val_s = max(valid_gt_smooth.max(), valid_pred_smooth.max())
axes_r2[1].plot([min_val_s, max_val_s], [min_val_s, max_val_s], 'k--', lw=1, label='y = x')
# Best fit line
m_s, c_s = np.polyfit(valid_gt_smooth, valid_pred_smooth, 1)
axes_r2[1].plot(valid_gt_smooth, m_s * valid_gt_smooth + c_s, 'r-', lw=2, label=f'Fit: y = {m_s:.2f}x + {c_s:.2f}')
axes_r2[1].set_title(f'Smoothed (Window={window})\n$R^2$ (Fit) = {r2_smooth:.4f}')
axes_r2[1].set_xlabel('Ground Truth EPR (Smoothed)')
axes_r2[1].set_ylabel('Predicted EPR (Smoothed)')
axes_r2[1].legend()

plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_r2_scatter_fit.png', dpi=150)
plt.show()


## 6. Spatial EP Map & Ensemble Average

In [ ]:
#
# Visualize Local EP Map 2D Heatmaps
#
model.eval()

test_sampler_one = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, 1,
    device=opt.device, train=False)
ens_idx, traj_idx = next(test_sampler_one)
b0 = ens_idx.to(test_video.device)
slices = [test_video[(b0, traj_idx[i].to(test_video.device))] for i in range(opt.seq_len)]
x = transform(torch.cat(slices, dim=1).float().to(opt.device))

with torch.no_grad():
    maps = model(x, return_maps=True) / opt.scalar # [B, K+1, Lx, Ly]

sample_idx = 0
dt = kwargs['dt']
vol = kwargs['Lx'] * kwargs['Ly'] * kwargs['dx']**2
phi_t = x[sample_idx, 0].cpu().numpy() # [Lx, Ly]
pred_map_k = maps[sample_idx].cpu().numpy() / (vol * dt) # [K+1, Lx, Ly]
pred_total_map = pred_map_k.sum(axis=0) # [Lx, Ly]
gt_map = gt_epr_maps[0]

fig_phi, ax_phi = plt.subplots(1, 1, figsize=(5, 4))
im_phi = ax_phi.imshow(phi_t.T, origin='lower', cmap='viridis')
ax_phi.set_title('Input State $\phi$')
fig_phi.colorbar(im_phi, ax=ax_phi)
plt.tight_layout()
plt.savefig(f'{current_result_folder}/input_state_2d.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vmax = max(np.abs(gt_map).max(), np.abs(pred_total_map).max(), 1e-12)
im1 = axes[0].imshow(gt_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('GT EPR Map')
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(pred_total_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Predicted Total EP Map')
fig.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map_2d.png', dpi=150)
plt.show()

# Plot individual k maps
distances_list = list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))
cols = min(len(distances_list), 6)
rows = int(np.ceil(len(distances_list) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
for idx, k in enumerate(distances_list if "distances_list" in locals() else list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))):
    vmax_k = max(np.abs(pred_map_k[idx]).max(), 1e-12)
    vmax_k = max(np.abs(pred_map_k[idx]).max(), 1e-12)
    im = axes[idx].imshow(pred_map_k[idx].T, origin='lower', cmap='RdBu_r', vmin=-vmax_k, vmax=vmax_k)
    axes[idx].set_title(f'k={k}')
    fig.colorbar(im, ax=axes[idx], shrink=0.6)
for i in range(len(distances_list), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map_2d_k.png', dpi=150)
plt.show()


In [ ]:
#
# Ensemble Averaged EP Map
#
print('[INFO] Calculating Ensemble Averaged Map...')
all_maps = []
test_sampler_batch = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in tqdm(test_sampler_batch):
        b0 = batch[0].to(test_video.device)
        vslices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))
        m = model(vx, return_maps=True) / opt.scalar # [B, K+1, Lx, Ly]
        all_maps.append(m.cpu().numpy())

dt = kwargs['dt']
vol = kwargs['Lx'] * kwargs['Ly'] * kwargs['dx']**2
all_maps = np.concatenate(all_maps, axis=0) / (vol * dt) # [N, K+1, Lx, Ly]
ensemble_map_k = all_maps.mean(axis=0) # [K+1, Lx, Ly]
ensemble_pred_total = ensemble_map_k.sum(axis=0) # [Lx, Ly]
ensemble_gt = gt_epr_maps[0] # [Lx, Ly]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vmax = max(np.abs(ensemble_gt).max(), np.abs(ensemble_pred_total).max(), 1e-12)

im0 = axes[0].imshow(ensemble_gt.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('GT Ensemble Mean EPR Map')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(ensemble_pred_total.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Predicted Ensemble Total EP Map')
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d.png', dpi=150)
plt.show()

# Plot individual k ensemble maps
distances_list = list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))
cols = min(len(distances_list), 6)
rows = int(np.ceil(len(distances_list) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
for idx, k in enumerate(distances_list if "distances_list" in locals() else list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))):
    vmax_k = max(np.abs(ensemble_map_k[idx]).max(), 1e-12)
    vmax_k = max(np.abs(ensemble_map_k[idx]).max(), 1e-12)
    im = axes[idx].imshow(ensemble_map_k[idx].T, origin='lower', cmap='RdBu_r', vmin=-vmax_k, vmax=vmax_k)
    axes[idx].set_title(f'k={k}')
    fig.colorbar(im, ax=axes[idx], shrink=0.6)
for i in range(len(distances_list), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d_k.png', dpi=150)
plt.show()


## 7. EP Spectrum — EP decomposition by correlation distance

각 거리 $k$에서의 평균 EP 기여도 $\langle J_k \rangle$를 시각화합니다.

In [ ]:
#
# EP spectrum: mean J_k for each distance k
#
model.eval()
all_J = []  # collect [B, K+1] arrays
test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J = model(x) / opt.scalar  # [B, K+1]
        all_J.append(J.cpu().numpy())

all_J = np.concatenate(all_J, axis=0)  # [N_total, K+1]
mean_J = all_J.mean(axis=0)             # [K+1]
std_J  = all_J.std(axis=0)              # [K+1]

distances = np.array(distances_list if "distances_list" in locals() else list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: mean EP at each distance
axes[0].bar(distances, mean_J / kwargs['dt'], yerr=std_J / kwargs['dt'] / np.sqrt(len(all_J)),
            capsize=3, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Kernel $k$')
axes[0].set_ylabel('$\\langle \\dot{S}_k \\rangle$')
axes[0].set_title('EP Spectrum: EP rate by correlation distance')
axes[0].set_xticks(distances)

# Cumulative sum
cum_J = np.cumsum(mean_J)
axes[1].plot(distances, cum_J / kwargs['dt'], 'o-', color='darkorange')
axes[1].set_xlabel('Kernel $k$')
axes[1].set_ylabel('$\\sum_{k} \\langle \\dot{S}_k \\rangle$')
axes[1].set_title('Cumulative EP rate')
axes[1].set_xticks(distances)

plt.tight_layout()
plt.savefig(f'{current_result_folder}/ep_spectrum_2d.png', dpi=150)
plt.show()

print(f'Total estimated EP rate: {mean_J.sum() / kwargs["dt"]:.6e}')
for idx, k in enumerate(distances_list if "distances_list" in locals() else list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))):
    print(f'  k={k}: J_k / dt = {mean_J[idx] / kwargs["dt"]:.6e}  '
          f'({100 * mean_J[idx] / mean_J.sum():.1f}%)')
